# Steuerung — KI-gestützte Dokumentenaufbereitung

**Referenz-Implementierung · Niveau DQR 5/6**

Dieses Notebook ist der **einzige Einstiegspunkt** der Pipeline. Die Module
`schema.py`, `layout.py`, `erkennung.py`, `stapel.py`, `kanonisch.py` und
`sichtung.py` liegen flach daneben und werden von hier importiert, nie
direkt ausgeführt.

## Warum ein Notebook und keine Skripte

Relative Pfade wie `befunde/Buch` werden gegen das **Arbeitsverzeichnis**
aufgelöst, und wer das setzt, hängt am Werkzeug:

| Start über | Arbeitsverzeichnis |
|---|---|
| Jupyter-Kernel | Ordner der Notebook-Datei |
| VS Code, Python-Datei | Workspace-Ordner (oft eine Ebene höher) |

Der Kernel verhält sich ohne jede Einstellung richtig. Deshalb steuert das
Notebook, und die Module bleiben frei von Pfadlogik.

**Wer eine `.py`-Datei doch direkt startet, bekommt das alte Verhalten
zurück.** Abschnitt 0 macht das sichtbar, statt es stillschweigend
zurechtzurücken.

## Ablauf

| Abschnitt | Inhalt | Modell |
|---|---|---|
| 0 | Umgebung prüfen | – |
| 1 | Stufe 1+2: Layout und Erkennung über ein Buch | ONNX + VLM |
| 2 | Sichtung: was steht in den Befunden? | – |
| 3 | Nachlauf für abgeschnittene Blöcke | VLM |
| 4 | Stufe 4a: kanonisches Dokument und Markdown | – |
| 5 | Kontrolle | – |

---
## 0. Umgebung prüfen

Erst nachsehen, dann rechnen. Diese Zelle stellt nichts richtig, sie meldet
nur — ein falsches Arbeitsverzeichnis soll auffallen und nicht kaschiert
werden.

In [ ]:
from pathlib import Path
import sys

# --- Anzupassen
BUCH = "Buch"
PDF  = Path("data/raw") / "Buch.pdf"
ONNX = Path("models") / "pp_doclayoutv3.onnx"
BEFUNDE = Path("data/interim/befunde")
AUSSCHNITTE = Path("data/interim/ausschnitte")
LMS  = "http://localhost:1234/v1"

BEFUNDE = Path("befunde")
AUSSCHNITTE = Path("ausschnitte")

print("Arbeitsverzeichnis:", Path.cwd())
print("Python            :", sys.version.split()[0], "\n")

for beschriftung, pfad in [("PDF", PDF), ("ONNX-Modell", ONNX),
                           ("Befunde", BEFUNDE / BUCH),
                           ("Ausschnitte", AUSSCHNITTE / BUCH)]:
    zustand = "vorhanden" if pfad.exists() else "fehlt"
    zusatz = ""
    if pfad.is_dir():
        zusatz = f"  ({len(list(pfad.glob('*'))) } Einträge)"
    print(f"  {beschriftung:14s} {zustand:10s} {pfad}{zusatz}")

# Liegt eine Ablage versehentlich eine Ebene höher? Häufigster Fall, wenn
# eine .py-Datei direkt gestartet wurde.
for name in ("befunde", "ausschnitte", "dokumente", "md"):
    fremd = Path("..") / name
    if fremd.exists() and not (Path.cwd() / name).exists():
        print(f"  ! {name!r} liegt eine Ebene höher: {fremd.resolve()}")

In [ ]:
# Module laden. autoreload, damit Änderungen an den .py-Dateien sofort
# wirken, ohne den Kernel neu zu starten.
%load_ext autoreload
%autoreload 2

import schema, layout, erkennung, stapel, kanonisch, sichtung
from schema import Stufe, Strom, befund_laden, befund_pfad

print("Schema-Version:", schema.SCHEMA_VERSION)
print("Klassenabbildung:", schema.selbsttest_docling())

In [ ]:
# Ist in LM Studio das richtige Modell geladen? Ein winziges Bild klärt in
# einer Sekunde, was ein Buchlauf sonst auf jeder Seite wiederholt.
from erkennung import Erkenner

ERKENNER = Erkenner(url=LMS)
print(ERKENNER.pruefe() if hasattr(ERKENNER, "pruefe")
      else f"Modell: {ERKENNER.modell_id}")

---
## 1. Stufe 1 und 2 über ein ganzes Buch

Der Lauf ist **stufenweise**: erst alle Seiten durch die Layout-Erkennung,
dann alle durch das VLM. Detektor und Sprachmodell sind damit nie
gleichzeitig geladen, die ONNX-Sitzung wird einmal geöffnet.

Fertige Seiten werden übersprungen (A2), gescheiterte übersprungen und
vermerkt (A3), Zeit und Modellversion je Seite festgehalten (A4).

**Erwartete Dauer:** Stufe 1 rund 0,4 s je Seite, Stufe 2 rund 11 s je
Seite. Für 222 Seiten also anderthalb Minuten plus vierzig.

In [ ]:
# Erst nur das Layout - kostet Sekunden und braucht kein LM Studio.
lauf = stapel.verarbeite_buch(PDF, buch=BUCH, bis=Stufe.LAYOUT,
                              onnx=ONNX, wurzel=BEFUNDE,
                              ausschnitt_wurzel=AUSSCHNITTE)

In [ ]:
# Kontrolle vor dem teuren Teil: eine Seite als Overlay ansehen.
from IPython.display import Image, display
import cv2

SEITE = 0
befund = befund_laden(befund_pfad(BEFUNDE, BUCH, SEITE))
bild, _ = layout.seite_rendern(PDF, seite=SEITE)

Path("data/interim/kontrolle").mkdir(exist_ok=True)
ziel = Path("data/interim/kontrolle") / f"{BUCH}_{SEITE:04d}.png"

farben = {Strom.HAUPT: (9, 105, 218), Strom.MARGINALIE: (23, 138, 63),
          Strom.BOILERPLATE: (130, 130, 130), Strom.APPARAT: (191, 121, 15)}
leinwand = bild.copy()[:, :, ::-1].copy()
for blk in befund.bloecke:
    bx, f = blk.bbox, farben[blk.strom]
    cv2.rectangle(leinwand, (int(bx.x0), int(bx.y0)), (int(bx.x1), int(bx.y1)),
                  (f[2], f[1], f[0]), 2)
    cv2.putText(leinwand, f"{blk.lese_index}:{blk.pp_label}",
                (int(bx.x0) + 3, max(14, int(bx.y0) - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (f[2], f[1], f[0]), 1, cv2.LINE_AA)
cv2.imwrite(str(ziel), leinwand)
display(Image(str(ziel), width=760))

In [ ]:
# Jetzt die Erkennung. Bei einem ganzen Buch: Laufzeit einplanen.
lauf = stapel.verarbeite_buch(PDF, buch=BUCH, bis=Stufe.ERKANNT,
                              onnx=ONNX, wurzel=BEFUNDE,
                              ausschnitt_wurzel=AUSSCHNITTE,
                              erkenner=ERKENNER, zeige_fortschritt=False)
print(lauf.zusammenfassung())

In [ ]:
# Was ist noch offen? Beantwortet A2, ohne etwas zu rechnen.
offen = stapel.offene_seiten(PDF, buch=BUCH, bis=Stufe.ERKANNT, wurzel=BEFUNDE)
print(len(offen), "offene Seiten", offen[:20])

---
## 2. Sichtung

Liest ausschließlich. Beantwortet mit Zahlen aus dem eigenen Korpus, was
sonst Vermutung bliebe: welche Klassen vorkommen, ob OTSL parsebar ist,
wie oft Detektionen ineinanderliegen, wie viele Seiten Überschriften
tragen.

In [ ]:
befunde = sichtung.befunde_lesen(BEFUNDE, BUCH)
sichtung.sichten(befunde)

---
## 3. Nachlauf für abgeschnittene Blöcke

`max_tokens` ist eine Sicherung gegen Wiederholungsschleifen, aber ein zu
knapper Deckel schneidet **still** ab: die Ausgabe liest sich flüssig und
hört einfach früher auf. Das Feld `finish_reason` der Antwort verrät es,
und die Erkennung schreibt es seit der Korrektur als Warnung in den Befund.

Diese Zelle sucht Blöcke, die verdächtig nahe am Deckel liegen, und lässt
nur deren Seiten erneut laufen — Minuten statt einer weiteren Stunde.

In [ ]:
GRENZE = 2800     # Zeichen; 1024 Tokens sind für Deutsch grob 3200

lang = [(len(b.text), bf.seite, b.id, b.pp_label, b.text[-50:].replace("\n", " "))
        for bf in befunde for b in bf.bloecke
        if b.text and len(b.text) > GRENZE]

for laenge, seite, bid, label, ende in sorted(lang, reverse=True)[:15]:
    print(f"{laenge:5d} S{seite:3d} #{bid:2d} {label:16s} …{ende!r}")

# Enden sie mitten im Wort, ist es der Deckel und kein Textmerkmal.
verdaechtig = sorted({s for _, s, _, _, _ in lang})
print(f"\n{len(lang)} Blöcke über {GRENZE} Zeichen auf {len(verdaechtig)} Seiten")
print(verdaechtig)

In [ ]:
# Nachlauf. neu=True ist nötig, sonst überspringt A2 genau diese Seiten.
if verdaechtig:
    bericht = stapel.stufe2_lauf(PDF, BUCH, verdaechtig, ERKENNER,
                                 wurzel=BEFUNDE, ausschnitt_wurzel=AUSSCHNITTE,
                                 neu=True, zeige_fortschritt=False)
    print(bericht.zeile())

# Ist danach noch etwas abgeschnitten?
uebrig = [(bf.seite, w) for bf in sichtung.befunde_lesen(BEFUNDE, BUCH)
          for w in bf.warnungen if "Token-Deckel" in w]
print(f"\n{len(uebrig)} Blöcke weiterhin am Deckel")
for seite, w in uebrig[:10]:
    print(f"  S{seite:3d}  {w}")

---
## 4. Stufe 4a — das kanonische Dokument

Übergang vom Arbeitsformat (Beobachtung, mit Widersprüchen) zum
`DoclingDocument` (Entscheidung: eine Region, ein Label, ein Text).
Vollständig deterministisch, ohne Sprachmodell.

Seitengrenzen werden nirgends aufgehoben: was auf einer Seite steht, bleibt
dort, damit jede Aussage rückwärts einer Seite zurechenbar bleibt.

In [ ]:
dok, bericht = kanonisch.buch_umwandeln(
    BUCH, wurzel=BEFUNDE,
    ziel_json=Path("data/processed/dokumente") / f"{BUCH}.json",
    ziel_md=Path("data/processed/md") / f"{BUCH}.md")

---
## 5. Kontrolle

Die billigste Prüfung ist ein Blick auf das Ergebnis. Wenn sich der
Fließtext an einer Kapitelgrenze flüssig liest und die Gliederungsebenen
stimmen, sind Lesereihenfolge, Erkennung und Übergang zugleich in Ordnung.

In [ ]:
# Gliederung: tragen die Ebenen?
from docling_core.types.doc import DocItemLabel

for item, _ in dok.iterate_items():
    if item.label is DocItemLabel.SECTION_HEADER:
        print(f"{'  ' * (item.level - 1)}{item.level}  {item.text[:70]}")

In [ ]:
# Ein Ausschnitt des Markdown, ab einer beliebigen Stelle.
text = (Path("data/processed/md") / f"{BUCH}.md").read_text(encoding="utf-8")
print(f"{len(text)} Zeichen, {text.count(chr(10)) + 1} Zeilen\n")
print(text[:3000])

In [ ]:
# Offene Bildunterschriften: die Arbeitsliste für den VLM-Pass in 4b.
print(len(bericht.offene_unterschriften), "offene Zuordnungen")
for seite, uid, oids in bericht.offene_unterschriften[:15]:
    bf = befund_laden(befund_pfad(BEFUNDE, BUCH, seite))
    u = next(b for b in bf.bloecke if b.id == uid)
    print(f"  S{seite:3d}  #{uid:2d} {(u.text or '')[:52]!r}  -> Kandidaten {oids}")

In [ ]:
# Alle Warnungen des Übergangs.
print("\n".join(bericht.warnungen) or "keine")

---
## Nicht in dieser Etappe

| # | Punkt | Warum |
|---|---|---|
| A7 | Absatz über Seitengrenze zusammenfassen | Seitengrenzen bleiben bestehen, damit die Seitenzuweisung erhalten bleibt |
| A13 | Inline-Formel zurück in den Satz | die Position im erkannten Text ist nicht bekannt |
| 4b | Bildunterschriften zuordnen, Kästen typisieren, Zahlentabellen zusammenfassen | keine eindeutig richtige Form; braucht ein Bildmodell |

Die Trennung ist der Kern: Deterministisches und Generatives
auseinanderzuhalten hält den prüfbaren Teil prüfbar.